# PCA dynamics for the 5-unit network — v8

At five hidden units the state is no longer directly plottable, so:

1. Take a trained 5-unit network.
2. Run the test set and record hidden activity: neurons x time x trials.
3. Reshape to neurons x (time.trials).
4. Fit PCA and keep the top 2 components.
5. Plot each trial's trajectory in that 2D PCA space, recreating the 2-unit figures (1, 2, 3b) plus a
   seed comparison.

The axes are now **principal components**, not raw neurons, and the decision-region backdrop is an
approximation (a 2D slice through the 5D readout at the PCA mean). Loads `./generated_trials_v8`.

## 1. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path

torch.manual_seed(0); np.random.seed(0)
device = torch.device("cpu")
DATA_DIR = Path("./generated_trials_v8"); OUT_DIR = Path("./pca_v8"); OUT_DIR.mkdir(exist_ok=True)
HIDDEN = 5; T_ON, T_OFF = 10, 20
CLASS_NAMES  = ["no det (0)", "det (1)", "right (2)", "left (3)"]
CLASS_COLORS = ["#cfcfcf", "#e0852e", "#c0392b", "#2e6ca4"]

## 2. Load data

In [ ]:
def load(split):
    d = np.load(DATA_DIR / f"{split}.npz", allow_pickle=True)
    out = {"X": d["X"].astype(np.float32), "y": d["y_4class"].astype(np.int64), "types": d["types"]}
    if "aud_int" in d:
        out["aud_int"] = d["aud_int"].astype(np.float32); out["vis_int"] = d["vis_int"].astype(np.float32)
    return out
train, test = load("train"), load("test")
T = test["X"].shape[2]; N = test["X"].shape[0]
SUBTASKS = sorted(set(test["types"]))
CONFLICT = ["det_multisensory", "loc_conflict_audL_visR", "loc_conflict_audR_visL"]
NONCONF  = [s for s in SUBTASKS if s not in CONFLICT]
print("Train", train["X"].shape, " Test", test["X"].shape)

## 3. Model (5 hidden units)

In [ ]:
class UnifiedGRU(nn.Module):
    def __init__(self, n_channels=4, hidden_size=5, n_classes=4):
        super().__init__()
        self.gru = nn.GRU(n_channels, hidden_size, batch_first=True)
        self.readout = nn.Linear(hidden_size, n_classes)
    def forward(self, x):
        x = x.transpose(1, 2); h, _ = self.gru(x); return self.readout(h)

## 4. Train (all seeds kept; no quality selection)

In [ ]:
def train_model(seed, n_epochs=50, lr=1e-3, batch=64):
    torch.manual_seed(seed); np.random.seed(seed)
    model = UnifiedGRU(4, HIDDEN, 4).to(device)
    loader = DataLoader(TensorDataset(torch.from_numpy(train["X"]), torch.from_numpy(train["y"])),
                        batch_size=batch, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=lr); loss_fn = nn.CrossEntropyLoss()
    for _ in range(n_epochs):
        model.train()
        for Xb, yb in loader:
            logits = model(Xb); B, Tt, C = logits.shape
            loss = loss_fn(logits.reshape(B*Tt, C), yb.unsqueeze(1).expand(B, Tt).reshape(B*Tt))
            opt.zero_grad(); loss.backward(); opt.step()
    return model

def evaluate(model):
    model.eval()
    with torch.no_grad(): pred = model(torch.from_numpy(test["X"]))[:, -1, :].argmax(-1).numpy()
    accs = {s: float((pred[test["types"]==s]==test["y"][test["types"]==s]).mean()) for s in SUBTASKS}
    return pred, accs, np.mean([accs[s] for s in NONCONF])

SEEDS = [0, 1, 2, 3, 4]
models = {}
for s in SEEDS:
    m = train_model(s); models[s] = m
    _, _, nc = evaluate(m); print("seed %d  non-conflict %.3f" % (s, nc))
DISPLAY_SEED = SEEDS[0]          # seed shown for figures 1-3b; set to any value in SEEDS
model = models[DISPLAY_SEED]

## 5. Record hidden activity and fit PCA

Steps 2-4 of the recipe. Hidden activity is recorded as (trials, time, neurons); we stack it to
(time.trials, neurons), fit a 2-component PCA on the pooled activity so every subtask shares one frame,
then project each trial back to (trials, time, 2).

In [ ]:
def hidden_activity(model):
    model.eval()
    with torch.no_grad():
        h, _ = model.gru(torch.from_numpy(test["X"]).transpose(1, 2))   # (N, T, HIDDEN)
    return h.numpy()

class PCA2:
    def __init__(self, X):                       # X: (n_samples, n_features)
        self.mean = X.mean(0)
        U, S, Vt = np.linalg.svd(X - self.mean, full_matrices=False)
        self.Vt = Vt[:2]                          # (2, features)
        self.evr = (S**2) / np.sum(S**2)
    def transform(self, X): return (X - self.mean) @ self.Vt.T
    def inverse(self, Z):  return Z @ self.Vt + self.mean

H = hidden_activity(model)                        # (N, T, 5)
pca = PCA2(H.reshape(N*T, HIDDEN))
Z = pca.transform(H.reshape(N*T, HIDDEN)).reshape(N, T, 2)   # (N, T, 2) PC trajectories
pred_all = model(torch.from_numpy(test["X"]))[:, -1, :].argmax(-1).detach().numpy()
print("PC1, PC2 explained variance: %.2f, %.2f  (sum %.2f)" % (pca.evr[0], pca.evr[1], pca.evr[:2].sum()))

## 6. Decision regions in PC space (approximate) and helpers

In [ ]:
Wro = model.readout.weight.detach().numpy(); bro = model.readout.bias.detach().numpy()
pad = 0.15
x0, x1 = Z[:,:,0].min()-pad, Z[:,:,0].max()+pad
y0, y1 = Z[:,:,1].min()-pad, Z[:,:,1].max()+pad
GXr = np.linspace(x0, x1, 300); GYr = np.linspace(y0, y1, 300)
GX, GY = np.meshgrid(GXr, GYr); _flat = np.stack([GX.ravel(), GY.ravel()], 1)
_grid5 = pca.inverse(_flat)                        # back to 5D at the PCA mean
_region = np.argmax(_grid5 @ Wro.T + bro, 1).reshape(GX.shape)

def draw_regions(ax, fill=True):
    if fill: ax.pcolormesh(GX, GY, _region, cmap=ListedColormap(CLASS_COLORS), alpha=0.20, shading="auto", vmin=0, vmax=3)
    ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2"); ax.set_xlim(x0, x1); ax.set_ylim(y0, y1)
def label_regions(ax, fs=11):
    r = _region.ravel()
    for cls in range(4):
        m = r == cls
        if m.sum() < 200: continue
        ax.text(_flat[m,0].mean(), _flat[m,1].mean(), CLASS_NAMES[cls], ha="center", va="center",
                fontsize=fs, fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.25", fc="white", ec=CLASS_COLORS[cls], lw=1.5, alpha=0.85), zorder=20)
def start_pt(): return Z[:, 0, :].mean(0)          # mean first-timestep PC (near-common start)

## Figure 1 (PC space) — average trajectory per subtask

In [ ]:
fig, ax = plt.subplots(figsize=(8.6, 8)); draw_regions(ax); label_regions(ax)
cols = plt.cm.tab20(np.linspace(0, 1, 20)); ci = 0
for s in NONCONF:
    tr = Z[test["types"] == s].mean(0); c = cols[ci]; ci += 1
    ax.plot(tr[:,0], tr[:,1], color=c, lw=2, label=s); ax.scatter(*tr[-1], color=c, s=40, marker="*", edgecolor="k", lw=0.4, zorder=6)
for s in CONFLICT:
    labs = [("det",1),("no-det",0)] if s == "det_multisensory" else [("right",2),("left",3)]
    for nm, lb in labs:
        idx = np.where((test["types"] == s) & (pred_all == lb))[0]
        if len(idx) == 0: continue
        tr = Z[idx].mean(0); c = cols[ci % 20]; ci += 1
        ax.plot(tr[:,0], tr[:,1], color=c, lw=2.4, ls="--", label="%s->%s" % (s, nm)); ax.scatter(*tr[-1], color=c, s=55, marker="*", edgecolor="k", lw=0.5, zorder=6)
ax.set_title("Figure 1 (PCA): average trajectory per subtask, 5-unit net (seed %d)" % DISPLAY_SEED)
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=7.5, frameon=False)
plt.tight_layout(); plt.savefig(OUT_DIR/"pca_fig1.png", dpi=150, bbox_inches="tight"); plt.show()

## Figure 2 (PC space) — conflict trajectories split by output label

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7.5)); draw_regions(ax); label_regions(ax)
ci = 0
for s in CONFLICT:
    labs = [("det",1),("no-det",0)] if s == "det_multisensory" else [("right",2),("left",3)]
    for nm, lb in labs:
        idx = np.where((test["types"] == s) & (pred_all == lb))[0]
        if len(idx) == 0: continue
        tr = Z[idx].mean(0); c = cols[ci % 20]; ci += 1
        ax.plot(tr[:,0], tr[:,1], color=c, lw=2.6, label="%s->%s (n=%d)" % (s, nm, len(idx)))
        ax.scatter(*tr[T_OFF], color=c, s=30, zorder=5); ax.scatter(*tr[-1], color=c, s=70, marker="*", edgecolor="k", lw=0.5, zorder=6)
ax.set_title("Figure 2 (PCA): conflict trajectories, split by output label (seed %d)" % DISPLAY_SEED)
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=8, frameon=False)
plt.tight_layout(); plt.savefig(OUT_DIR/"pca_fig2.png", dpi=150, bbox_inches="tight"); plt.show()

## Figure 3b (PC space) — average conflict trajectory per signed-gap band, by conflict type

In [ ]:
N_BINS = 6; LOC = ["loc_conflict_audL_visR", "loc_conflict_audR_visL"]; cmap = plt.cm.coolwarm
allg = np.abs(test["aud_int"][np.isin(test["types"], LOC)] - test["vis_int"][np.isin(test["types"], LOC)])
vmax = allg.max()
fig, axes = plt.subplots(1, 2, figsize=(15, 7))
for ax, ctype in zip(axes, LOC):
    idx = np.where(test["types"] == ctype)[0]
    gap = np.abs(test["aud_int"][idx] - test["vis_int"][idx])
    signed = np.where(test["y"][idx] == 2, gap, -gap)
    edges = np.quantile(signed, np.linspace(0, 1, N_BINS+1)); edges[-1] += 1e-9
    b = np.clip(np.digitize(signed, edges[1:-1]), 0, N_BINS-1)
    draw_regions(ax); label_regions(ax)
    for k in range(N_BINS):
        sel = b == k
        if sel.sum() == 0: continue
        tr = Z[idx[sel]].mean(0); g = signed[sel].mean()
        c = cmap(0.5 + 0.5*g/(vmax+1e-9)); side = "right" if g > 0 else "left"
        ax.plot(tr[:,0], tr[:,1], color=c, lw=2.6, label="gap %+.2f -> %s (n=%d)" % (g, side, sel.sum()))
        ax.scatter(*tr[-1], color=c, s=85, marker="*", edgecolor="k", lw=0.6, zorder=6)
    ax.set_title(ctype); ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), fontsize=8, frameon=False, ncol=2)
sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(-vmax, vmax)); sm.set_array([])
fig.colorbar(sm, ax=axes, label="signed intensity gap  (- left wins / + right wins)", fraction=0.025)
fig.suptitle("Figure 3b (PCA): conflict trajectory per signed-gap band, 5-unit net (seed %d)" % DISPLAY_SEED, fontsize=13)
plt.savefig(OUT_DIR/"pca_fig3b.png", dpi=150, bbox_inches="tight"); plt.show()

## Figure 5 (PC space) — seed comparison as small multiples

At 5 units, each seed has its **own** PCA frame, so overlaying seeds on shared axes is not meaningful.
Instead we show one subtask's average trajectory for each seed in that seed's own PC space, side by side,
which is the fairest way to compare whether the networks behave similarly.

In [ ]:
SUBTASK_FOR_SEEDS = "loc_conflict_audL_visR"
fig, axes = plt.subplots(1, len(SEEDS), figsize=(4*len(SEEDS), 4.2))
for ax, s in zip(np.atleast_1d(axes), SEEDS):
    m = models[s]; Hs = hidden_activity(m); pcs = PCA2(Hs.reshape(N*T, HIDDEN))
    Zs = pcs.transform(Hs.reshape(N*T, HIDDEN)).reshape(N, T, 2)
    idx = np.where(test["types"] == SUBTASK_FOR_SEEDS)[0]
    tr = Zs[idx].mean(0)
    ax.plot(tr[:,0], tr[:,1], color="k", lw=2); ax.scatter(*tr[0], color="k", s=25); ax.scatter(*tr[-1], color="crimson", s=60, marker="*", edgecolor="k", lw=0.5)
    ax.set_title("seed %d (evr %.2f)" % (s, pcs.evr[:2].sum()), fontsize=10); ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
fig.suptitle("Figure 5 (PCA): '%s' averaged trajectory, each seed in its own PC space" % SUBTASK_FOR_SEEDS, fontsize=12)
plt.tight_layout(rect=[0,0,1,0.94]); plt.savefig(OUT_DIR/"pca_fig5_seeds.png", dpi=150, bbox_inches="tight"); plt.show()

## Notes

- Cell 5 prints how much variance the top 2 PCs capture. If it is low, the 2D view is missing structure and
  a third component would be needed.
- The decision-region backdrop is a slice through the 5D readout at the PCA mean, so it is indicative rather
  than exact; a trajectory can appear to cross a boundary because its off-plane components differ from the mean.
- PCA gives a more comparable frame than the raw hidden axes, but each seed's frame is still its own (free up
  to sign and rotation), which is why figure 5 uses small multiples rather than an overlay.